# Echocardiography EDA: EV9V, MIMIC-IV-Echo and EchoNet-Dynamic

Read-only review of a frozen local analysis. Raw sources and row-level clinical records stay local. MIMIC predicted views are not expert labels.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT if (ROOT / 'inventory_summary.json').exists() else ROOT / 'outputs/eda_2026-10-05_v3'
assert (OUT / 'inventory_summary.json').exists(), 'Set OUT to the local EDA package directory'


## 0-1. Scope, source tables and units
One inventory row is one native file. EV9V JPEG directories are derivatives of its videos; frames are correlated observations.

In [ ]:
summary = pd.read_csv(OUT / 'tables/dataset_summary.csv')
display(summary)
with open(OUT / 'inventory_summary.json') as f:
    snapshot = json.load(f)
print({k: snapshot[k] for k in ['started_utc', 'rows_by_dataset', 'table_rows', 'table_columns']})


## 2-3. Inclusion, timing and missingness
Timing missingness uses dynamic objects as its denominator. Absent expert labels and patient mappings are structural information gaps, not values to impute.

In [ ]:
display(pd.read_csv(OUT / 'tables/information_availability.csv'))
display(pd.read_csv(OUT / 'tables/format_resolution_distribution.csv'))


## 4. Distribution figures and examples
Inspect captions alongside every figure. Temporal examples are candidates; no independent expert boundaries were added.

In [ ]:
figures = json.loads((OUT / 'figure_manifest.json').read_text())
for item in figures:
    print(item['name'], '\n', item['caption'])
    display(Image(filename=str(OUT / 'figures' / (item['name'] + '.png')), width=1000))


## Quantitative checks
File counts and timing do not establish pixel-level completeness or anatomical accuracy.

In [ ]:
display(pd.read_csv(OUT / 'tables/bmode_duration_summary.csv'))
qc = pd.read_csv(OUT / 'tables/qc_sample_results.csv')
display(qc.groupby('dataset').agg(sampled_videos=('case_id','size'), decoded_frames=('n_decoded','sum')))
assert qc.decode_error.isna().all()


## Reproduce
Run from the repository root with the project virtual environment. The English report narrative is specific to the frozen 2026-10-05 inventory and verifies its hash. To regenerate that report:

```bash
.venv/bin/python scripts/eda_report.py --out outputs/eda_2026-10-05_v3
```

For a new inventory and descriptive analysis, choose a new output directory. New downloads can change the counts; review and update the report narrative before using it with a different snapshot.

```bash
.venv/bin/python scripts/eda_inventory.py --out outputs/eda_new
.venv/bin/python scripts/eda_analyze.py --out outputs/eda_new
```
The report dependency is available through the `eda` project extra; the `dev` extra supplies notebook dependencies. These scripts reuse existing model caches and do not run training.